In [1]:
import ibis 

con = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="postgres",
    port=5432,
    database="my_db",
)

tbl_name = "air_traffic"


In [2]:
con.sql(f"SELECT * FROM {tbl_name} LIMIT 10").execute()

,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Enplaned,Other,Terminal 2,D,1198
5,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Deplaned,Other,Terminal 1,B,24124
6,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Enplaned,Other,Terminal 1,B,23613
7,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Deplaned,Other,Terminal 2,D,4983
8,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Enplaned,Other,Terminal 2,D,4604
9,1999,1999-07-01,Air Europe,PE,Air Europe,PE,International,Europe,Deplaned,Other,Terminal 2,D,205


In [3]:
import sys
import os

# Add project root to path
# Get the directory of this file, then go up one level to project root
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)


In [4]:
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con = con, tbl_name = tbl_name)

schema = tbl_attr.schema
print(schema)


Year bigint, Date timestamp without time zone, Operating Airline character varying, Operating Airline IATA Code character varying, Published Airline character varying, Published Airline IATA Code character varying, GEO Summary character varying, GEO Region character varying, Activity Type Code character varying, Price Category Code character varying, Terminal character varying, Boarding Area character varying, Passenger Count bigint


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [6]:
system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format.
Please ensure that the field names in the query are enclosed in double quotes.
CREATE TABLE {tbl_name} ({schema})
""".strip()


user_template = "Write a SQL query that returns: {question}"

messages = [("system", system_template), ("user", user_template)]

prompt_template = ChatPromptTemplate.from_messages(messages)


In [7]:
base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"
# model = "gpt-5.2"
llm = ChatOpenAI(
    base_url=base_url, 
    api_key=api_key, 
    temperature=0, 
    model= model
)


In [8]:
chain = prompt_template | llm

question = "How many rows are in the table?"

In [9]:
llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
        }
    )

In [10]:
query = llm_output.content
print(query)

SELECT COUNT(*) FROM air_traffic;


In [11]:
con.sql(query).execute()

,count
0,38546


In [12]:
def basic_sql_agent(chain, question, tbl_name, schema, con):
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
        }
    )
    query = llm_output.content
    print(query)
    output = con.sql(query).execute()
    return output


In [13]:
basic_sql_agent(chain, 
                question = "how many passengers landed during 2024?", 
                tbl_name=tbl_name,
                schema = schema,
                con = con)

SELECT SUM("Passenger Count") AS "Total Passengers Landed"
FROM air_traffic
WHERE "Activity Type Code" = 'Deplaned'
AND EXTRACT(YEAR FROM "Date") = 2024;


,Total Passengers Landed
0,26079194


## Adding Additional Context 

In [15]:
basic_sql_agent(
    chain,
    question="how many passengers departed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    con=con,
)


SELECT SUM("Passenger Count") AS "Total Passengers Departed"
FROM "air_traffic"
WHERE "Activity Type Code" = 'Departure' AND EXTRACT(YEAR FROM "Date") = 2024;


,Total Passengers Departed
0,None


In [16]:
basic_sql_agent(
    chain,
    question="What are the unique values of the 'Activity Type Code' field?",
    tbl_name=tbl_name,
    schema=schema,
    con=con,
)


SELECT DISTINCT "Activity Type Code" FROM air_traffic;


,Activity Type Code
0,Deplaned
1,Thru / Transit
2,Enplaned


In [22]:
system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format.
Please ensure that the field names in the query are enclosed in double quotes.

{additional_context}

CREATE TABLE {tbl_name} ({schema})

""".strip()


user_template = "Write a SQL query that returns: {question}"

messages = [("system", system_template), ("user", user_template)]

prompt_template = ChatPromptTemplate.from_messages(messages)

chain = prompt_template | llm


In [23]:
def basic_sql_agent(chain, question, tbl_name, schema,con, additional_context = ""):
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
            "additional_context": additional_context
        }
    )
    query = llm_output.content
    print(query)
    output = con.sql(query).execute()
    return output


In [24]:
basic_sql_agent(
    chain,
    question="how many passengers departed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context = "The 'Activity Type Code' unique values are: 'Deplaned', 'Enplaned', and 'Thru / Transit'",
    con=con,
)


SELECT SUM("Passenger Count") AS "Total Departed Passengers"
FROM air_traffic
WHERE "Activity Type Code" = 'Enplaned' AND EXTRACT(YEAR FROM "Date") = 2024;


,Total Departed Passengers
0,26054586


In [ ]:
from sql_ai_agent.db_handler import get_character_distinct_values
from sql_ai_agent.prompt_handler import format_distinct_values_for_prompt

distinct_values = get_character_distinct_values(con = con, tbl_schema = tbl_attr, tbl_name = tbl_name)

print(distinct_values)

{'Operating Airline': ['ABC Aerolineas S.A. de C.V. dba Interjet', 'Aer Lingus, Ltd.', 'Aeroflot Russian International Airlines', 'Aeromexico', 'Air 2000', 'Air Atlanta Icelandic', 'Air Berlin', 'Air Canada', 'Air Canada Jazz', 'Air China', 'Air Europe', 'Air France', 'Air India Limited', 'Air Italy S.P.A', 'Air New Zealand', 'Air Pacific Limited dba Fiji Airways', 'Air Premia, Inc.', 'AirTran Airways', 'Air Transat', 'Air Wisconsin', 'Alaska Airlines', 'Alitalia Airlines', 'Allegiant Air', 'Allegro Airlines', 'All Nippon Company Airways, Ltd.', 'American Airlines', 'American Eagle Airlines', 'Ameriflight', 'Asiana Airlines', 'ATA Airlines', 'Atlantic Southeast Airlines', 'Atlas Air, Inc', 'BelAir Airlines', 'Boeing Company', 'Breeze Aviation Group, Inc.', 'British Airways', 'Canadian Airlines', 'Casino Express', 'Cathay Pacific', 'Champion Air', 'China Airlines', 'China Eastern', 'China Eastern Airlines, Inc', 'China Southern', 'Comair', 'Compass Airlines', 'Condor Flugdienst GmbH', '

In [31]:
distinct_values_formatted = format_distinct_values_for_prompt(distinct_values)
print(distinct_values_formatted)

The following columns have known categorical values:
- "Operating Airline": 'ABC Aerolineas S.A. de C.V. dba Interjet', 'Aer Lingus, Ltd.', 'Aeroflot Russian International Airlines', 'Aeromexico', 'Air 2000', 'Air Atlanta Icelandic', 'Air Berlin', 'Air Canada', 'Air Canada Jazz', 'Air China' ...
- "Operating Airline IATA Code": '4O', '4T', '5Y', '9W', 'A8', 'AA', 'AB', 'AC', 'AF', 'AI' ...
- "Published Airline": 'ABC Aerolineas S.A. de C.V. dba Interjet', 'Aer Lingus, Ltd.', 'Aeroflot Russian International Airlines', 'Aeromexico', 'Air 2000', 'Air Atlanta Icelandic', 'Air Berlin', 'Air Canada', 'Air China', 'Air Europe' ...
- "Published Airline IATA Code": '4O', '4T', '5Y', '9W', 'A8', 'AA', 'AB', 'AC', 'AF', 'AI' ...
- "GEO Summary": 'Domestic', 'International'
- "GEO Region": 'Asia', 'Australia / Oceania', 'Canada', 'Central America', 'Europe', 'Mexico', 'Middle East', 'South America', 'US'
- "Activity Type Code": 'Deplaned', 'Enplaned', 'Thru / Transit'
- "Price Category Code": 'Low

In [32]:
basic_sql_agent(
    chain,
    question="how many passengers departed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    con=con,
)


SELECT SUM("Passenger Count") AS "Total Passengers Departed"
FROM air_traffic
WHERE "Activity Type Code" = 'Enplaned' AND EXTRACT(YEAR FROM "Date") = 2024;


,Total Passengers Departed
0,26054586


In [34]:
def basic_sql_agent(chain, question, tbl_name, schema, con, additional_context=""):
    distinct_values = get_character_distinct_values(
        con=con, tbl_schema=tbl_attr, tbl_name=tbl_name
    )
    distinct_values_formatted = format_distinct_values_for_prompt(distinct_values)
    additional_context = additional_context + "\n" + distinct_values_formatted 
  
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
            "additional_context": additional_context,
        }
    )
    query = llm_output.content
    print(query)
    output = con.sql(query).execute()
    return output


In [35]:
basic_sql_agent(
    chain,
    question="how many passengers were transiting during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context="",
    con=con,
)


SELECT SUM("Passenger Count") AS "Total Passengers"
FROM air_traffic
WHERE "Activity Type Code" = 'Thru / Transit' AND EXTRACT(YEAR FROM "Date") = 2024;


,Total Passengers
0,77159


## Adding Memory

In [36]:
basic_sql_agent(
    chain,
    question="how many passengers departed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    con=con,
)


SELECT SUM("Passenger Count") AS "Total Passengers Departed"
FROM air_traffic
WHERE "Activity Type Code" = 'Enplaned' AND EXTRACT(YEAR FROM "Date") = 2024;


,Total Passengers Departed
0,26054586


In [37]:
basic_sql_agent(
    chain,
    question="And arriving?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    con=con,
)


SELECT "Year", "Date", "Operating Airline", "Operating Airline IATA Code", "Published Airline", "Published Airline IATA Code", "GEO Summary", "GEO Region", "Activity Type Code", "Price Category Code", "Terminal", "Boarding Area", "Passenger Count"
FROM air_traffic
WHERE "Activity Type Code" = 'Deplaned';


,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,1324
2,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Deplaned,Other,Terminal 1,B,24124
3,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Deplaned,Other,Terminal 2,D,4983
4,1999,1999-07-01,Air Europe,PE,Air Europe,PE,International,Europe,Deplaned,Other,Terminal 2,D,205
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17991,2025,2025-07-01,United Airlines,UA,United Airlines,UA,International,Mexico,Deplaned,Other,International,G,42417
17992,2025,2025-07-01,Vietnam Airlines JSC,VN,Vietnam Airlines JSC,VN,International,Asia,Deplaned,Other,International,A,4490
17993,2025,2025-07-01,Virgin Atlantic,VS,Virgin Atlantic,VS,International,Europe,Deplaned,Other,International,A,14654
17994,2025,2025-07-01,WestJet,WS,WestJet,WS,International,Canada,Deplaned,Other,International,A,14451


In [40]:
memory = []

previous_conversion = "Previous conversions: \n" + ", ".join(memory)
print(previous_conversion)


Previous conversions: 



In [15]:
import pandas as pd

df = pd.read_csv(project_root + "/data/air_traffic_gold.csv")

df.head()


,Unnamed: 0,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,3,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,4,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,1198


In [16]:
con_db = ibis.duckdb.connect()
con_db.create_table(tbl_name,df, overwrite=True)


DatabaseTable: memory.main.air_traffic
  Unnamed: 0                  int64
  Year                        int64
  Date                        string
  Operating Airline           string
  Operating Airline IATA Code string
  Published Airline           string
  Published Airline IATA Code string
  GEO Summary                 string
  GEO Region                  string
  Activity Type Code          string
  Price Category Code         string
  Terminal                    string
  Boarding Area               string
  Passenger Count             int64

In [32]:
basic_sql_agent(
    chain,
    question="Could you please describe the fields in the database and their attributes?",
    tbl_name=tbl_name,
    schema=schema,
    con=con_db,
)


TypeError: basic_sql_agent() missing 1 required positional argument: 'database'

In [ ]:
basic_sql_agent(
    chain,
    question="how many passengers landed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    con=con_db,
)


SELECT SUM("Passenger Count") AS "Total Passengers Landed"
FROM air_traffic
WHERE "Activity Type Code" = 'Deplaned'
AND EXTRACT(YEAR FROM "Date") = 2024;


BinderException: Binder Error: No function matches the given name and argument types 'date_part(STRING_LITERAL, VARCHAR)'. You might need to add explicit type casts.
	Candidate functions:
	date_part(VARCHAR[], DATE) -> STRUCT()
	date_part(VARCHAR[], INTERVAL) -> STRUCT()
	date_part(VARCHAR[], TIME) -> STRUCT()
	date_part(VARCHAR[], TIMESTAMP) -> STRUCT()
	date_part(VARCHAR[], TIME WITH TIME ZONE) -> STRUCT()
	date_part(VARCHAR[], TIME_NS) -> STRUCT()
	date_part(VARCHAR, DATE) -> BIGINT
	date_part(VARCHAR, INTERVAL) -> BIGINT
	date_part(VARCHAR, TIME) -> BIGINT
	date_part(VARCHAR, TIMESTAMP) -> BIGINT
	date_part(VARCHAR, TIME WITH TIME ZONE) -> BIGINT
	date_part(VARCHAR, TIME_NS) -> BIGINT
	date_part(VARCHAR[], TIMESTAMP WITH TIME ZONE) -> STRUCT()
	date_part(VARCHAR, TIMESTAMP WITH TIME ZONE) -> BIGINT


LINE 4: AND EXTRACT(YEAR FROM "Date") = 2024;
            ^

In [19]:
query = """
SELECT SUM("Passenger Count") AS "Total Passengers Landed"
FROM air_traffic
WHERE "Activity Type Code" = 'Deplaned'
AND EXTRACT(YEAR FROM "Date") = 2024;
"""

In [20]:
con.sql(query).execute()

,Total Passengers Landed
0,26079194


In [21]:
con_db.sql(query).execute()


BinderException: Binder Error: No function matches the given name and argument types 'date_part(STRING_LITERAL, VARCHAR)'. You might need to add explicit type casts.
	Candidate functions:
	date_part(VARCHAR[], DATE) -> STRUCT()
	date_part(VARCHAR[], INTERVAL) -> STRUCT()
	date_part(VARCHAR[], TIME) -> STRUCT()
	date_part(VARCHAR[], TIMESTAMP) -> STRUCT()
	date_part(VARCHAR[], TIME WITH TIME ZONE) -> STRUCT()
	date_part(VARCHAR[], TIME_NS) -> STRUCT()
	date_part(VARCHAR, DATE) -> BIGINT
	date_part(VARCHAR, INTERVAL) -> BIGINT
	date_part(VARCHAR, TIME) -> BIGINT
	date_part(VARCHAR, TIMESTAMP) -> BIGINT
	date_part(VARCHAR, TIME WITH TIME ZONE) -> BIGINT
	date_part(VARCHAR, TIME_NS) -> BIGINT
	date_part(VARCHAR[], TIMESTAMP WITH TIME ZONE) -> STRUCT()
	date_part(VARCHAR, TIMESTAMP WITH TIME ZONE) -> BIGINT


LINE 5: AND EXTRACT(YEAR FROM "Date") = 2024;
            ^

## Why did it Errored?

This comes down to differences in how **Postgres** and **DuckDB** handle data types and date extraction, not the aggregate logic itself.

Your query is valid SQL for Postgres, but DuckDB is stricter about what `EXTRACT` can operate on.

SQL has nuances and dialects, and, as we saw before, a query that works in one database may fail in another. To make our SQL AI agent more robust, we can simply instruct the LLM to return the query according to the database we are working with. Let’s update the prompt.



In [33]:
system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format.
Important: I am querying the data against a {database} database, please make sure the returned query works with {database} SQL dialects. 
Please ensure that the field names in the query are enclosed in double quotes.
CREATE TABLE {tbl_name} ({schema})
""".strip()


In [34]:
user_template = "Write a SQL query that returns: {question}"
messages = [("system", system_template), ("user", user_template)]
prompt_template = ChatPromptTemplate.from_messages(messages)
chain = prompt_template | llm

In [35]:
def basic_sql_agent(chain, question, tbl_name, schema, database, con):
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
            "database": database
        }
    )
    query = llm_output.content
    print(query)
    output = con.sql(query).execute()
    return output


In [36]:
basic_sql_agent(
    chain,
    question="how many passengers landed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    database="DuckDB",
    con=con_db,
)


SELECT SUM("Passenger Count") AS "Total Passengers"
FROM "air_traffic"
WHERE "Activity Type Code" = 'Deplaned'
AND EXTRACT(YEAR FROM "Date") = 2024;


BinderException: Binder Error: No function matches the given name and argument types 'date_part(STRING_LITERAL, VARCHAR)'. You might need to add explicit type casts.
	Candidate functions:
	date_part(VARCHAR[], DATE) -> STRUCT()
	date_part(VARCHAR[], INTERVAL) -> STRUCT()
	date_part(VARCHAR[], TIME) -> STRUCT()
	date_part(VARCHAR[], TIMESTAMP) -> STRUCT()
	date_part(VARCHAR[], TIME WITH TIME ZONE) -> STRUCT()
	date_part(VARCHAR[], TIME_NS) -> STRUCT()
	date_part(VARCHAR, DATE) -> BIGINT
	date_part(VARCHAR, INTERVAL) -> BIGINT
	date_part(VARCHAR, TIME) -> BIGINT
	date_part(VARCHAR, TIMESTAMP) -> BIGINT
	date_part(VARCHAR, TIME WITH TIME ZONE) -> BIGINT
	date_part(VARCHAR, TIME_NS) -> BIGINT
	date_part(VARCHAR[], TIMESTAMP WITH TIME ZONE) -> STRUCT()
	date_part(VARCHAR, TIMESTAMP WITH TIME ZONE) -> BIGINT


LINE 4: AND EXTRACT(YEAR FROM "Date") = 2024;
            ^

In [25]:
basic_sql_agent(
    chain,
    question="how many passengers landed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    database="PostgreSQL",
    con=con,
)



SELECT SUM("Passenger Count") AS "Total Passengers"
FROM "air_traffic"
WHERE "Activity Type Code" = 'Deplaned'
AND EXTRACT(YEAR FROM "Date") = 2024;


,Total Passengers
0,26079194


In [27]:
schema

'Year bigint, Date timestamp without time zone, Operating Airline character varying, Operating Airline IATA Code character varying, Published Airline character varying, Published Airline IATA Code character varying, GEO Summary character varying, GEO Region character varying, Activity Type Code character varying, Price Category Code character varying, Terminal character varying, Boarding Area character varying, Passenger Count bigint'